In [10]:
pip install torch torchaudio transformers librosa soundfile numpy

Note: you may need to restart the kernel to use updated packages.


In [11]:
import torch
print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Torch version: 2.9.1+cpu
GPU available: False


In [12]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

MODEL_NAME = "Harveenchadha/vakyansh-wav2vec2-tamil-tam-250"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)

model.eval()

preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

D:\Anaconda\Anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saras\.cache\huggingface\hub\models--Harveenchadha--vakyansh-wav2vec2-tamil-tam-250. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

In [18]:
import librosa

audio_path = "audio_samples/SP25_KG_M_1.wav"

speech, sr = librosa.load(audio_path, sr=16000, mono=True)

print("Sample rate:", sr)
print("Audio length:", speech.shape)

Sample rate: 16000
Audio length: (463203,)


In [19]:
inputs = processor(
    speech,
    sampling_rate=16000,
    return_tensors="pt",
    padding=False
)

In [20]:
with torch.no_grad():
    logits = model(inputs.input_values).logits

In [21]:
import torch

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)

print("RAW TRANSCRIPTION:")
print(transcription[0])

RAW TRANSCRIPTION:
எனுங்க எந்தர சர இப்ப கெளம்பிட்டீங்கள பொத்த வாழத்துக்கு ஏனுங்க இருந்து சாப்பட்டு போலாம்லங்க சாப்பட்டீங்கா வச சேரிங்க நம்மளுக்கு பொத்த வாளத்துல நம வாயிலுங்க எந்த கோயிலுங்க புள்ளார் கோயில் சரிங்க நம்மு சாமி என்ன சாமிங்க ங்களுது வ்ய்கறயாங்களா அட அருபீங்கஅருபீங்க சரி சரிங்க அப்பம் பஸ்ட எத்தன மணி பேசு போறீங்கஙக சரி சரி சரி சரிங்க அப்பறம் அ்உங்க ப்பாரு எங்க இருக்கிறாருங்க எந்த ஊருக் போயிருக்கிறாருங்க அப்படிங்களா


In [22]:
slang_map = {
    "என்னடா": "என்ன டா",
    "வாடா": "வா டா",
    "போடா": "போ டா",
    "டேய்": "டேய்",
    "மச்சி": "மச்சி",
    "செம்ம": "செம்ம"
}

def reinforce_slang(text):
    for k, v in slang_map.items():
        text = text.replace(k, v)
    return text

final_text = reinforce_slang(transcription[0])

print("FINAL SLANG-AWARE OUTPUT:")
print(final_text)

FINAL SLANG-AWARE OUTPUT:
எனுங்க எந்தர சர இப்ப கெளம்பிட்டீங்கள பொத்த வாழத்துக்கு ஏனுங்க இருந்து சாப்பட்டு போலாம்லங்க சாப்பட்டீங்கா வச சேரிங்க நம்மளுக்கு பொத்த வாளத்துல நம வாயிலுங்க எந்த கோயிலுங்க புள்ளார் கோயில் சரிங்க நம்மு சாமி என்ன சாமிங்க ங்களுது வ்ய்கறயாங்களா அட அருபீங்கஅருபீங்க சரி சரிங்க அப்பம் பஸ்ட எத்தன மணி பேசு போறீங்கஙக சரி சரி சரி சரிங்க அப்பறம் அ்உங்க ப்பாரு எங்க இருக்கிறாருங்க எந்த ஊருக் போயிருக்கிறாருங்க அப்படிங்களா
